# ДЗ-18 · Этап 1 (Colab): 7B-модели — quantized vs full precision

Здесь сравниваем «тяжёлые» модели на **Google Colab T4 GPU**:
* **Mistral-7B-Instruct** и **Llama-2-7B-chat** (gated — нужен доступ + `HF_TOKEN`);
* **4-bit квантование** (`bitsandbytes`, QLoRA-стиль) vs **full precision** (fp16);
* метрики: tokens/sec, пик **VRAM**, качество извлечения на CUAD.

> Запускать в Colab: `Runtime → Change runtime type → T4 GPU`. `bitsandbytes` требует CUDA и локально на Windows без NVIDIA не работает — поэтому этот этап вынесен в Colab.

In [ ]:
!nvidia-smi -L
!pip -q install -U transformers accelerate bitsandbytes datasets sentencepiece

In [ ]:
# Загрузи рядом наши модули или склонируй репозиторий, чтобы переиспользовать
# промпт и парсер (build_messages / parse_entities) и загрузку CUAD.
# from google.colab import files; files.upload()   # ie_extractor.py, data_prep.py, evaluate.py
import os
# os.environ['HF_TOKEN'] = 'hf_...'   # нужен для gated Llama-2
from ie_extractor import build_messages, parse_entities, ENTITY_TYPES
from data_prep import get_subset
from evaluate import evaluate, format_report

## Универсальный загрузчик: 4-bit quantized или full (fp16)
Один и тот же код, разница только в `BitsAndBytesConfig`.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_model(name, quantized=True):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    kw = dict(device_map='auto')
    if quantized:
        kw['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw['torch_dtype'] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    return tok, model

In [ ]:
@torch.no_grad()
def extract_one(tok, model, text, max_new_tokens=384):
    prompt = tok.apply_chat_template(build_messages(text), tokenize=False,
                                     add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    dt = time.perf_counter() - t0
    new = out[0][inputs['input_ids'].shape[1]:]
    txt = tok.decode(new, skip_special_tokens=True)
    return parse_entities(txt), {'gen_tokens': int(new.shape[0]), 'sec': dt,
                                 'tok/s': int(new.shape[0]) / dt if dt else 0}

In [ ]:
def benchmark_7b(name, quantized, samples):
    torch.cuda.reset_peak_memory_stats()
    tok, model = load_model(name, quantized=quantized)
    preds, tps = [], []
    for s in samples:
        p, st = extract_one(tok, model, s.text)
        preds.append(p); tps.append(st['tok/s'])
    metrics = evaluate(preds, [s.gold for s in samples])
    res = {'model': name.split('/')[-1], 'mode': '4bit' if quantized else 'fp16',
           'tok/s': round(sum(tps)/len(tps), 1),
           'VRAM_GB': round(torch.cuda.max_memory_allocated()/1e9, 2),
           'micro_F1': metrics['micro']['f1']}
    del model; torch.cuda.empty_cache()
    return res, metrics

## Данные

In [ ]:
samples = get_subset(n_docs=50, max_chars=2500)
print('контрактов:', len(samples))

## Прогон: Mistral-7B (4-bit vs fp16) и Llama-2-7B (4-bit)
fp16 7B на T4 (16 ГБ) помещается впритык; если падает по памяти — оставь только 4-bit.

In [ ]:
import pandas as pd
rows = []
MISTRAL = 'mistralai/Mistral-7B-Instruct-v0.2'
LLAMA   = 'meta-llama/Llama-2-7b-chat-hf'   # gated: прими условия + HF_TOKEN

r, _ = benchmark_7b(MISTRAL, quantized=True,  samples=samples); rows.append(r); print(r)
r, _ = benchmark_7b(MISTRAL, quantized=False, samples=samples); rows.append(r); print(r)
try:
    r, _ = benchmark_7b(LLAMA, quantized=True, samples=samples); rows.append(r); print(r)
except Exception as e:
    print('Llama-2 пропущена (нет доступа/HF_TOKEN):', e)

pd.DataFrame(rows)

## Выводы (этап 1)

* **4-bit квантование** уменьшает VRAM в ~3–4 раза (7B fp16 ≈ 14 ГБ → 4-bit ≈ 4–5 ГБ), позволяя 7B-модели работать на T4, с небольшой потерей качества.
* **full precision (fp16)** обычно даёт чуть выше F1, но требует больше VRAM и часто медленнее по tokens/sec на T4 из-за пропускной способности памяти.
* Сравнение с маленькими CPU-моделями (0.5B/1.5B) — в `ie_extraction.ipynb`.